# BME 574 — Project 2 starter
### Five steps. Five cells for you to write. Everything else is done.

You will write a nonlinear least-squares fitter from scratch, check it against a **certified**
answer, then hand the same job to `lmfit` and confirm the two agree. Finally you fit real
biomedical data and report parameters **with uncertainties**.

**Your job is the five cells marked `TODO`** — about twenty lines of code in total.

---

### What you hand in

1. **This notebook**, run top to bottom, all five `TODO` cells filled, every check passing.
2. **`REPORT.md`** — one page, the six questions in Step 6.
3. **`AI_USE.md`** — which AI tools you used and for what.

### Before you start

Do **`BME574_Fitting_explained.ipynb`** first (~30 min, read and run). It derives everything you
are about to implement. If you have not used NumPy, do the Project 1 primer before that.

### The one idea to hold on to

> **A fitted parameter without an uncertainty is not a result.** And a fit that converged is not
> necessarily right — nonlinear models can have more than one answer that fits equally well.
> Steps 4 and 5 are where this project is won or lost.

---
## Step 0 · Load your bundle

Your track comes as a single `.npz`. One bundle may hold several **curves** (one per subject,
per replicate, or per drug) that all share the same model.

| Key | What it is |
|---|---|
| `x`, `y` | the measurements, all curves stacked together |
| `curve_id` | which curve each point belongs to |
| `curve_const` | a per-curve constant the model needs, e.g. dose |
| `model_key` | which model to fit — the function is defined for you below |
| `param_names`, `p0` | the parameters, and a starting guess |
| `ref_values` | certified or published values to check against, where they exist |

Start with **`warmup.npz`** (the NIST problem) whatever your track is. Switch to your real track
at Step 5.

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import lmfit

BUNDLE = 'bundles/warmup.npz'   # <-- Step 5 switches this to your track
SEED = 0
rng = np.random.default_rng(SEED)
plt.rcParams.update({'font.size': 11, 'figure.dpi': 110})
np.set_printoptions(precision=6, suppress=True)

ok1 = ok2 = ok3 = ok4 = ok5 = False   # progress flags, set by the checks below
print('lmfit', lmfit.__version__)

In [ ]:
# ---- the models. These are given; you do not need to change them. ----

def m_boxbod(x, p, const=0.0):
    b1, b2 = p
    return b1 * (1.0 - np.exp(-b2 * x))

def m_biexp(x, p, const=0.0):
    A, alpha, B, beta = p
    return A * np.exp(-alpha * x) + B * np.exp(-beta * x)

def m_oral1c(x, p, const=1.0):
    ka, ke, V = p
    d = ka - ke
    d = np.where(np.abs(d) < 1e-12, 1e-12, d)
    return (const * ka / (V * d)) * (np.exp(-ke * x) - np.exp(-ka * x))

def m_hill(x, p, const=0.0):
    Emin, Emax, EC50, n = p
    xs = np.maximum(np.asarray(x, float), 1e-12)
    r = (xs / max(float(EC50), 1e-12)) ** n
    return Emin + (Emax - Emin) * r / (1.0 + r)

MODELS = {'boxbod': m_boxbod, 'biexp': m_biexp, 'oral1c': m_oral1c, 'hill': m_hill}


def load(path):
    D = np.load(path, allow_pickle=True)
    B = dict(x=D['x'], y=D['y'], curve_id=D['curve_id'],
             curve_names=[str(s) for s in D['curve_names']],
             curve_const=D['curve_const'], const_name=str(D['const_name']),
             model_key=str(D['model_key']),
             param_names=[str(s) for s in D['param_names']],
             p0=D['p0'], x_label=str(D['x_label']), y_label=str(D['y_label']),
             track=str(D['track']), ref_names=[str(s) for s in D['ref_names']],
             ref_values=D['ref_values'], ref_stderr=D['ref_stderr'],
             ref_source=str(D['ref_source']))
    B['model'] = MODELS[B['model_key']]
    return B


def curve(B, i):
    """Return (x, y, const) for curve i of bundle B."""
    m = B['curve_id'] == i
    return B['x'][m], B['y'][m], float(B['curve_const'][i])


B = load(BUNDLE)
print('=' * 64)
print(' ', B['track'])
print('=' * 64)
print(f"  model      {B['model_key']}   parameters: {', '.join(B['param_names'])}")
print(f"  curves     {len(B['curve_names'])}   points: {len(B['x'])}")
print(f"  start p0   {dict(zip(B['param_names'], B['p0']))}")
if len(B['ref_values']):
    print(f"  reference  {B['ref_source']}")

In [ ]:
# what does one curve look like?
nshow = min(6, len(B['curve_names']))
fig, axes = plt.subplots(1, nshow, figsize=(2.4 * nshow, 2.6), squeeze=False)
for j, ax in enumerate(axes[0]):
    xs, ys, _ = curve(B, j)
    o = np.argsort(xs)
    ax.plot(xs[o], ys[o], 'o-', ms=4, lw=.9)
    ax.set_title(B['curve_names'][j], fontsize=8)
fig.supxlabel(B['x_label'], fontsize=9); fig.supylabel(B['y_label'], fontsize=9)
fig.tight_layout(); plt.show()

---
## Step 1 · **TODO 1** — residuals, and a Jacobian you have checked

Two things.

**`residuals(p, x, y, const)`** — what the model gets wrong at each point. One line:
`model(x, p, const) - y`.

**`jacobian_fd(p, x, const)`** — the derivative of each prediction with respect to each
parameter, by central differences. One column per parameter:

```python
(model(x, p + step, const) - model(x, p - step, const)) / (2 * step[j])
```

We use finite differences rather than hand-derived formulas so that the same code works for every
track. The check below confirms it against an exact analytic Jacobian for the warm-up model.

> **Why this comes first.** A wrong derivative does not raise an error. It gives you a fitter that
> converges slowly, or to the wrong place, and you will blame the algorithm. Check it before you
> write anything else.

In [ ]:
MODEL = B['model']

def residuals(p, x, y, const=0.0):
    # ---- TODO 1a ------------------------------------------------------
    r = None
    # --------------------------------------------------------------------
    return r


def jacobian_fd(p, x, const=0.0, eps=1e-6):
    p = np.asarray(p, float)
    cols = []
    for j in range(len(p)):
        step = np.zeros_like(p)
        step[j] = eps * max(abs(p[j]), 1.0)
        # ---- TODO 1b ---------------------------------------------------
        col = None
        # -----------------------------------------------------------------
        cols.append(col)
    if any(c is None for c in cols):
        return None
    return np.column_stack(cols)


def check(name, cond, hint=''):
    print(f'  {"OK  " if cond else "NOT YET"}  {name}')
    if not cond and hint:
        print(f'           hint: {hint}')
    return bool(cond)


xs, ys, cs = curve(B, 0)
p_try = np.asarray(B['p0'], float)
r_try = residuals(p_try, xs, ys, cs)
J_try = jacobian_fd(p_try, xs, cs)

checks = [check('residuals returns one value per data point',
                isinstance(r_try, np.ndarray) and r_try.shape == ys.shape,
                'MODEL(x, p, const) - y'),
          check('residuals have the right sign convention',
                isinstance(r_try, np.ndarray) and r_try.shape == ys.shape
                and np.allclose(r_try, MODEL(xs, p_try, cs) - ys),
                'model minus data, not data minus model'),
          check('Jacobian is n_points x n_parameters',
                isinstance(J_try, np.ndarray) and J_try.shape == (len(xs), len(p_try)),
                'one column per parameter, stacked with np.column_stack')]

# independent check: for the warm-up model we know the exact derivative
if B['model_key'] == 'boxbod' and isinstance(J_try, np.ndarray):
    b1, b2 = p_try
    e = np.exp(-b2 * xs)
    J_exact = np.column_stack([1 - e, b1 * xs * e])
    checks.append(check('finite-difference Jacobian matches the analytic one',
                        np.abs(J_try - J_exact).max() < 1e-5,
                        'check the +/- step and the division by 2*step[j]'))

ok1 = all(checks)
print()
print('Step 1 complete.' if ok1 else 'Fix the cells above before continuing.')

---
## Step 2 · **TODO 2** — Levenberg–Marquardt

Three lines inside the loop.

```python
A = J.T @ J + lam * np.eye(len(p))    # damped normal equations
g = -J.T @ r
step = np.linalg.solve(A, g)
```

Everything else — the accept/reject logic that raises `lam` after a bad step and lowers it after a
good one — is written for you. Drop the `lam * np.eye(...)` term and you have Gauss–Newton, which
the explainer notebook showed diverging from this very starting point.

In [ ]:
def cost(p, x, y, const=0.0):
    r = residuals(p, x, y, const)
    return 0.5 * float((r ** 2).sum())


def lm_fit(p0, x, y, const=0.0, n_iter=300, lam=1e-2):
    """Levenberg-Marquardt. Returns (params, history_of_cost)."""
    p = np.array(p0, float)
    c = cost(p, x, y, const)
    hist = [c]
    for _ in range(n_iter):
        r = residuals(p, x, y, const)
        J = jacobian_fd(p, x, const)
        if not (np.all(np.isfinite(r)) and np.all(np.isfinite(J))):
            break
        # ---- TODO 2 ----------------------------------------------------
        A = None      # J.T @ J  plus  lam * identity
        g = None      # -J.T @ r
        step = None   # solve A @ step = g
        # -----------------------------------------------------------------
        if step is None:
            raise SystemExit('TODO 2 is not filled in yet.')
        p_new = p + step
        c_new = cost(p_new, x, y, const)
        if np.isfinite(c_new) and c_new < c:
            p, c = p_new, c_new          # accept: trust the linear model more
            lam = max(lam * 0.3, 1e-12)
            hist.append(c)
            if np.linalg.norm(step) < 1e-12 * (1 + np.linalg.norm(p)):
                break
        else:
            lam *= 10                    # reject: trust it less
            if lam > 1e12:
                break
    return p, np.array(hist)


if not ok1:
    raise SystemExit('Finish TODO 1 first.')

p_hat, hist = lm_fit(B['p0'], xs, ys, cs)
print('fitted parameters:', dict(zip(B['param_names'], np.round(p_hat, 6))))
print('RSS              :', 2 * cost(p_hat, xs, ys, cs))
print('accepted steps   :', len(hist) - 1)

ok2 = all([
    check('the fit returned finite parameters',
          np.all(np.isfinite(p_hat))),
    check('the cost went down',
          len(hist) > 1 and hist[-1] < hist[0],
          'if the cost never improved, check the sign of g'),
])

if ok2 and len(B['ref_values']) and len(B['ref_values']) == len(p_hat):
    rel = np.abs(p_hat - B['ref_values']) / np.abs(B['ref_values'])
    print()
    print(f"{'':10s} {'yours':>16s} {'reference':>16s} {'rel err':>10s}")
    for nm, a, b_, e in zip(B['param_names'], p_hat, B['ref_values'], rel):
        print(f'{nm:10s} {a:16.8f} {b_:16.8f} {e:10.2e}')
    ok2 = ok2 and check('matches the certified / true values',
                        rel.max() < 1e-4 if B['model_key'] == 'boxbod' else rel.max() < 0.3)

### The convergence figure *(written for you)* — **Figure 1**

In [ ]:
if not ok2:
    raise SystemExit('Finish TODO 2 first.')

fig, ax = plt.subplots(1, 2, figsize=(11.5, 4))
ax[0].semilogy(2 * np.array(hist), 'o-', ms=4)
ax[0].set(xlabel='accepted step', ylabel='residual sum of squares',
          title='Figure 1a — convergence')

o = np.argsort(xs)
xg = np.linspace(xs.min(), xs.max(), 300)
ax[1].plot(xs[o], ys[o], 'o', ms=6, label='data')
ax[1].plot(xg, MODEL(xg, p_hat, cs), '-', lw=1.8, label='your fit')
ax[1].set(xlabel=B['x_label'], ylabel=B['y_label'], title='Figure 1b — the fit')
ax[1].legend(fontsize=9)
fig.tight_layout(); plt.show()

### A stress test *(written for you)* — does your fitter survive a bad guess?

NIST ships **two** official starting points for this problem. The bundle uses the easy one so
that everything above runs smoothly. Here is the hard one.

In the explainer notebook, Gauss–Newton diverged to infinity from this point in a single step.
Your damped version should still get there. So should `lmfit` — but it does not, with its default
settings, which is worth seeing.

In [ ]:
if not ok2:
    raise SystemExit('Finish TODO 2 first.')


def _resid_for_lmfit(p, x, y, const):
    return MODEL(x, list(p.valuesdict().values()), const) - y


if B['model_key'] == 'boxbod':
    HARD = [1.0, 1.0]
    with np.errstate(over='ignore', invalid='ignore'):
        p_hard, hist_hard = lm_fit(HARD, xs, ys, cs)
    rel = np.abs(p_hard - B['ref_values']) / np.abs(B['ref_values'])
    shown = dict(zip(B['param_names'], np.round(p_hard, 6)))
    print(f'your LM from the hard start {HARD}:')
    print(f'   {shown}   max rel err {rel.max():.2e}')

    _p = lmfit.Parameters()
    for _n, _v in zip(B['param_names'], HARD):
        _p.add(_n, value=float(_v), min=0)
    _o = lmfit.minimize(_resid_for_lmfit, _p, args=(xs, ys, cs))
    _v = np.array([_o.params[n].value for n in B['param_names']])
    _r = np.abs(_v - B['ref_values']) / np.abs(B['ref_values'])
    shown2 = dict(zip(B['param_names'], np.round(_v, 6)))
    print('lmfit from the same start:')
    print(f'   {shown2}   max rel err {_r.max():.2e}')
    print()
    if rel.max() < 1e-6 < _r.max():
        print('Your fitter got there and the library did not.')
        print('Not because lmfit is worse - because its default damping schedule is tuned')
        print('for typical problems and this starting point is deliberately pathological.')
        print('The lesson is not to write your own. It is: ALWAYS check that a fit')
        print('from a second starting point lands in the same place. Step 5 does that.')
else:
    print('(stress test only applies to the warm-up bundle)')

---
## Step 3 · **TODO 3** — standard errors

A parameter without an uncertainty is not a result. The covariance of the fitted parameters is

$$ \mathrm{Cov} \approx s^2\,(\mathbf{J}^{\!\top}\mathbf{J})^{-1}, \qquad
   s^2 = \frac{\mathrm{RSS}}{n-p} $$

and the standard error of each parameter is the square root of the corresponding diagonal entry.
Three lines. `np.linalg.inv` and `np.diag` are what you need.

In [ ]:
if not ok2:
    raise SystemExit('Finish TODO 2 first - this step needs the fitted parameters.')


def covariance(p, x, y, const=0.0):
    J = jacobian_fd(p, x, const)
    n, npar = len(y), len(p)
    rss = 2 * cost(p, x, y, const)
    # ---- TODO 3 --------------------------------------------------------
    s2 = None      # rss divided by the degrees of freedom
    cov = None     # s2 * inverse of (J.T @ J)
    # ---------------------------------------------------------------------
    return cov


cov = covariance(p_hat, xs, ys, cs)
se = np.sqrt(np.diag(cov)) if cov is not None else None

ok3 = all([
    check('covariance is n_par x n_par',
          isinstance(cov, np.ndarray) and cov.shape == (len(p_hat), len(p_hat)),
          's2 * np.linalg.inv(J.T @ J)'),
    check('standard errors are positive and finite',
          se is not None and np.all(np.isfinite(se)) and np.all(se > 0),
          'did you divide by (n - n_parameters)?'),
])

if ok3:
    print()
    for nm, v, s in zip(B['param_names'], p_hat, se):
        print(f'  {nm:8s} = {v:12.6f}  +/- {s:10.6f}   ({100 * s / abs(v):5.1f}%)')
    if len(B['ref_stderr']) == len(se) and np.all(np.isfinite(B['ref_stderr'])):
        print()
        print('  certified standard errors:', np.round(B['ref_stderr'], 6))
        ok3 = ok3 and check('matches the certified standard errors',
                            np.abs(se - B['ref_stderr']).max() / np.abs(B['ref_stderr']).max() < 1e-3)

---
## Step 4 · **TODO 4** — now do it with `lmfit`, and compare

You have written the algorithm, so you know what it does. From here on use the library — it
handles scaling, convergence tests, bounds and a dozen edge cases you should not have to.

Build an `lmfit.Parameters` object and minimize. Two lines:

```python
for name, val in zip(B['param_names'], B['p0']):
    pars.add(name, value=val, min=0)         # these models have positive parameters

out = lmfit.minimize(resid_lmfit, pars, args=(xs, ys, cs))
```

> **The gotcha.** Inside the residual function, `p` holds `Parameter` objects, not numbers. Use
> `p.valuesdict()` to get them all at once — handing a `Parameter` straight to NumPy raises a
> confusing `__array__` error.

In [ ]:
def resid_lmfit(p, x, y, const):
    vals = list(p.valuesdict().values())
    return MODEL(x, vals, const) - y


pars = lmfit.Parameters()
# ---- TODO 4 ------------------------------------------------------------
# add one Parameter per entry of B['param_names'], starting at B['p0'],
# with a lower bound of 0; then minimize resid_lmfit.
out = None
# --------------------------------------------------------------------------

ok4 = check('lmfit returned a fit result', out is not None and hasattr(out, 'params'),
            "lmfit.minimize(resid_lmfit, pars, args=(xs, ys, cs))")

if ok4:
    lm_vals = np.array([out.params[n].value for n in B['param_names']])
    lm_se = np.array([out.params[n].stderr or np.nan for n in B['param_names']])
    print()
    print(f"{'':10s} {'yours':>14s} {'lmfit':>14s} {'your se':>12s} {'lmfit se':>12s}")
    for i, nm in enumerate(B['param_names']):
        print(f'{nm:10s} {p_hat[i]:14.6f} {lm_vals[i]:14.6f} {se[i]:12.6f} {lm_se[i]:12.6f}')
    agree = np.abs(p_hat - lm_vals) / np.maximum(np.abs(lm_vals), 1e-12)
    print()
    ok4 = ok4 and check('your fitter agrees with lmfit', agree.max() < 1e-3,
                        'if these disagree, one of you found a different minimum')
    print()
    print(lmfit.fit_report(out))

Read the `[[Correlations]]` block at the bottom of that report. Any pair above about 0.95 means
those two parameters are **not independently determined by your data** — the fit can trade one
against the other and barely change. That is an identifiability problem, and it belongs in your
memo.

---
## Step 5 · **TODO 5** — your real track, every curve

Switch `TRACK_BUNDLE` below to the file you were given, then fill in the loop: fit each curve and
collect the parameters and their standard errors.

Everything you need is already written — `lm_fit`, `covariance`, and `curve(B, i)`. Four lines.

In [ ]:
TRACK_BUNDLE = 'bundles/indometh.npz'   # <-- your track

if not ok4:
    raise SystemExit('Finish TODO 4 first.')
if not os.path.exists(TRACK_BUNDLE):
    raise SystemExit(f'{TRACK_BUNDLE} not found - ask for your track bundle.')

T = load(TRACK_BUNDLE)
MODEL = T['model']          # the helper functions above use this global
print(T['track'])
print(f"  {len(T['curve_names'])} curves, parameters: {', '.join(T['param_names'])}")

fits = []
for i in range(len(T['curve_names'])):
    xi, yi, ci = curve(T, i)
    # ---- TODO 5 --------------------------------------------------------
    pi = None        # fit this curve with lm_fit, starting from T['p0']
    covi = None      # its covariance
    # ----------------------------------------------------------------------
    if pi is None or covi is None:
        raise SystemExit('TODO 5 is not filled in yet.')
    fits.append(dict(name=T['curve_names'][i], p=np.asarray(pi),
                     se=np.sqrt(np.diag(covi)),
                     rss=2 * cost(np.asarray(pi), xi, yi, ci)))

P = np.array([f['p'] for f in fits])
SE = np.array([f['se'] for f in fits])

ok5 = all([
    check('one parameter set per curve', P.shape == (len(T['curve_names']), len(T['p0']))),
    check('all parameters finite', np.all(np.isfinite(P))),
])

if ok5:
    print()
    hdr = f"{'curve':<14s}" + ''.join(f'{n:>13s}' for n in T['param_names'])
    print(hdr); print('-' * len(hdr))
    for f in fits:
        print(f"{f['name'][:13]:<14s}" + ''.join(f'{v:13.4f}' for v in f['p']))
    print()
    print('across curves:')
    for j, n in enumerate(T['param_names']):
        print(f'  {n:8s} median {np.median(P[:, j]):10.4f}   '
              f'spread {P[:, j].min():9.4f} to {P[:, j].max():9.4f}')

### **Figure 2** — the fits, and **Figure 3** — parameter spread *(written for you)*

In [ ]:
if not ok5:
    raise SystemExit('Finish TODO 5 first.')

n = min(6, len(fits))
fig, axes = plt.subplots(2, n, figsize=(2.5 * n, 5), squeeze=False,
                         gridspec_kw=dict(height_ratios=[2.2, 1]))
for j in range(n):
    xi, yi, ci = curve(T, j)
    o = np.argsort(xi)
    xg = np.linspace(xi.min(), xi.max(), 300)
    axes[0][j].plot(xi[o], yi[o], 'o', ms=4)
    axes[0][j].plot(xg, MODEL(xg, fits[j]['p'], ci), '-', lw=1.5)
    axes[0][j].set_title(fits[j]['name'][:16], fontsize=8)
    res = MODEL(xi, fits[j]['p'], ci) - yi
    axes[1][j].axhline(0, color='k', lw=.6)
    axes[1][j].plot(xi[o], res[o], 'o', ms=3, color='#c0392b')
    axes[1][j].set_xlabel(T['x_label'], fontsize=7)
axes[0][0].set_ylabel(T['y_label'], fontsize=8)
axes[1][0].set_ylabel('residual', fontsize=8)
fig.suptitle('Figure 2 — fit (top) and residuals (bottom)', y=1.02)
fig.tight_layout(); plt.show()

print('Look at the residual row. If it curves, your model is wrong for that curve;')
print('random scatter about zero is what a good fit looks like.')

In [ ]:
if not ok5:
    raise SystemExit('Finish TODO 5 first.')

npar = len(T['param_names'])
fig, axes = plt.subplots(1, npar, figsize=(2.8 * npar, 3.2), squeeze=False)
for j, ax in enumerate(axes[0]):
    ax.errorbar(np.arange(len(P)), P[:, j], yerr=SE[:, j], fmt='o', ms=5, capsize=3)
    ax.axhline(np.median(P[:, j]), color='#c0392b', ls='--', lw=1)
    ax.set_title(T['param_names'][j], fontsize=10)
    ax.set_xlabel('curve', fontsize=8)
    if P[:, j].min() > 0 and P[:, j].max() / P[:, j].min() > 30:
        ax.set_yscale('log')
fig.suptitle('Figure 3 — parameter estimates with standard errors, across curves', y=1.04)
fig.tight_layout(); plt.show()

print('An error bar larger than the spread between subjects means your data constrain')
print('that parameter worse than biology varies it. Say so in the memo.')

### The multi-start check *(written for you)* — **run this, it is the point of the project**

Converging is not the same as being right. Restart each fit from several random starting points
and see whether you always land in the same place. If you do not, your data admit **more than one**
answer, and reporting a single parameter set would be misleading.

> **One subtlety, handled for you.** A sum of exponentials is unchanged if you swap its two
> terms — $(A,\alpha,B,\beta)$ and $(B,\beta,A,\alpha)$ describe the *same curve*. That is a
> relabelling, not a second answer, so the code below puts every solution in a standard order
> (fastest rate first) before comparing. Genuine ambiguity is what survives that.

In [ ]:
if not ok5:
    raise SystemExit('Finish TODO 5 first.')


def canonical(p, model_key):
    """Put a solution in a standard form before comparing two of them.

    A sum of exponentials is unchanged if you swap the two terms: (A, alpha, B, beta)
    and (B, beta, A, alpha) are THE SAME CURVE with the labels exchanged. Without
    fixing an order, a multi-start would report that as two different answers.
    We order by rate, fastest first. Same for absorption vs elimination.
    """
    p = np.asarray(p, float).copy()
    if model_key == 'biexp' and p[3] > p[1]:
        p = p[[2, 3, 0, 1]]
    return p


def multistart(T, i, n_start=12, spread=3.0, seed=0):
    g = np.random.default_rng(seed)
    xi, yi, ci = curve(T, i)
    sols = []
    for _ in range(n_start):
        start = np.asarray(T['p0'], float) * g.uniform(1 / spread, spread, len(T['p0']))
        try:
            with np.errstate(over='ignore', invalid='ignore'):
                p, _ = lm_fit(start, xi, yi, ci)
                c = 2 * cost(p, xi, yi, ci)
            if np.all(np.isfinite(p)) and np.isfinite(c):
                sols.append((c, canonical(p, T['model_key'])))
        except Exception:
            pass
    return sols


print('multi-start: do we always land in the same place?\n')
n_amb = 0
for i in range(min(4, len(fits))):
    sols = multistart(T, i)
    if not sols:
        print(f"  {T['curve_names'][i]:<14s} no start converged"); continue
    best = min(s[0] for s in sols)
    near = [p for c, p in sols if c < best * 1.01]      # equally good fits
    spread = (np.max(near, axis=0) - np.min(near, axis=0)) / np.abs(np.median(near, axis=0))
    amb = spread.max() > 0.05
    n_amb += amb
    print(f"  {T['curve_names'][i]:<14s} {len(sols):2d} converged, "
          f'{len(near):2d} equally good, max param spread among them '
          f"{spread.max():6.1%}  {'<-- AMBIGUOUS' if amb else ''}")
print()
if n_amb:
    print('At least one curve has MORE THAN ONE answer that fits equally well.')
    print('This must be in your memo. It is a finding, not a bug.')
else:
    print('All curves landed in one place from every start. Say so in the memo.')

---
## Step 6 · The memo

Write `REPORT.md`, **one page**, answering these six. Short answers, complete sentences.

1. **What model are you fitting, and what does each parameter mean physically?** Give units.
2. **Did your fitter agree with `lmfit`, and did both agree with the certified warm-up values?**
   Give the numbers.
3. **Report your parameters with uncertainties.** Say where the uncertainty came from and what
   assumption it rests on.
4. **Which parameters are poorly determined, and how do you know?** Point at the correlation
   block, the error bars in Figure 3, or the multi-start result.
5. **Did the multi-start find more than one answer?** If yes, say what distinguishes them and
   which you would report. If no, say that.
6. **What would make these estimates trustworthy enough to use?** More sampling times? A
   different experiment design? Fixing a parameter from prior knowledge? Be concrete.

> A project that reports "two parameter sets fit this data equally well, here is the evidence,
> and here is why the experiment cannot distinguish them" is a complete and excellent project.

In [ ]:
ok5 = bool(globals().get('ok5', False))

steps = {'Step 1  residuals and Jacobian': ok1,
         'Step 2  Levenberg-Marquardt': ok2,
         'Step 3  standard errors': ok3,
         'Step 4  lmfit agreement': ok4,
         'Step 5  all curves fitted': ok5}
for name, done in steps.items():
    print(f'  {"OK  " if done else "NOT YET"}  {name}')
print()
print('All steps pass. Write REPORT.md and AI_USE.md.' if all(steps.values())
      else 'Some steps are incomplete - scroll up to the ones marked NOT YET.')

---
## If you get stuck

| Symptom | Almost always |
|---|---|
| `LinAlgError: Singular matrix` | Two parameters are perfectly traded off. Raise the starting `lam`, or fix one parameter. |
| Cost never decreases | Sign error: `g` must be `-J.T @ r`. |
| `Parameter.__array__()` error | You passed an lmfit `Parameter` into NumPy. Use `p.valuesdict()`. |
| Standard errors are `nan` | `J.T @ J` is not invertible — the classic sign of an unidentifiable parameter. Report it. |
| Fit is perfect but parameters are absurd | Check units, and check that `curve_const` is being used. |
| Your fitter and lmfit disagree | You have found different minima. Run the multi-start; this is Q5. |

Bring the failing cell and its printed output to office hours.